# Sesión 3 — MBD

## Modelo lineal

Estudiamos la resistencia del hormigón (`Strength`) a partir de su composición y su edad.
El recorrido incluye modelos lineales, selección de variables, diagnóstico, transformaciones y predicción.

En VS Code, selecciona un kernel de Python y ejecuta las celdas en orden con **Shift + Enter**.
Abre **Variables** para inspeccionar las tablas y **Esquema / Outline** para navegar entre apartados.
Los controles bajo los ejemplos se activan al ejecutar sus celdas.

Puedes cambiar los valores de una celda y volver a ejecutarla. Para repetir todo el análisis,
usa **Reiniciar** y **Ejecutar todo**.
[Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Preparación

Usaremos NumPy, pandas, Matplotlib, SciPy, statsmodels e ipywidgets.
Si falta alguna librería en el kernel seleccionado, descomenta y ejecuta la siguiente línea una vez.

In [ ]:
# %pip install numpy pandas matplotlib scipy statsmodels ipywidgets threadpoolctl ipykernel

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import statsmodels.api as sm
from IPython.display import display
from pandas.plotting import scatter_matrix
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.special import boxcox
from statsmodels.nonparametric.smoothers_lowess import lowess
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})
pd.set_option("display.max_columns", 20)

## Lectura de datos y descriptiva

Leemos `Concrete_train.txt`, con cabecera y columnas separadas por tabuladores.
Localizamos la carpeta `datos` desde la carpeta de trabajo actual.

In [ ]:
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "Concrete_train.txt").is_file():
        DATA_PATH = carpeta / "datos"
        break
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

datos = pd.read_csv(DATA_PATH / "Concrete_train.txt", sep="\t")
datos.index = np.arange(1, len(datos) + 1)
datos.index.name = "observacion"
columnas_originales = list(datos.columns)
predictores = [columna for columna in datos if columna != "Strength"]

### Inspección

Las etiquetas de las filas van de 1 a n para identificar las observaciones del conjunto.
Usaremos `loc` para seleccionarlas por su etiqueta.

In [ ]:
datos.shape  # Número de filas y columnas

In [ ]:
datos

In [ ]:
datos.describe()

#### Diagramas de caja de todas las variables

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
_ = ax.boxplot([datos[columna].dropna() for columna in columnas_originales])
_ = ax.set_xticks(np.arange(1, len(columnas_originales) + 1))
_ = ax.set_xticklabels(columnas_originales, rotation=90)
fig.tight_layout()
plt.show()

#### Explorar todos los pares de variables

In [ ]:
_ = scatter_matrix(datos[columnas_originales], figsize=(14, 14), diagonal="hist", alpha=0.35, s=8)
plt.show()

### Relación entre resistencia y cemento

La curva LOWESS resume la relación sin imponer una recta.
Selecciona una variable y ajusta la fracción de observaciones usada para suavizar la curva.
Si hay muchos valores repetidos, ampliamos el vecindario cuando sea necesario y mostramos la fracción aplicada.

In [ ]:
def dibujar_relacion(ax, x, y, nombre, fraccion=2/3):
    ax.scatter(x, y, s=12, alpha=0.45)
    aplicada = fraccion
    while True:
        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always", RuntimeWarning)
            curva = lowess(y, x, frac=aplicada, it=3, delta=0.01 * np.ptp(x))
        numericos = any(issubclass(aviso.category, RuntimeWarning) for aviso in avisos)
        if np.isfinite(curva).all() and not numericos:
            break
        if aplicada >= 1:
            raise ValueError("No se puede estimar LOWESS con estos valores.")
        aplicada = min(1, aplicada + 0.05)
    ax.plot(curva[:, 0], curva[:, 1], color="red")
    ax.set(xlabel=nombre, ylabel="Strength")
    if aplicada > fraccion + 1e-8:
        ax.text(0.02, 0.98, f"Fracción aplicada: {aplicada:.2f}",
                transform=ax.transAxes, va="top", fontsize=9)

def explorar_relacion(variable, fraccion):
    fig, ax = plt.subplots()
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable, fraccion)
    ax.set_title("Resistencia y " + variable)
    plt.show()

In [ ]:
control_relacion = widgets.interactive(
    explorar_relacion,
    variable=widgets.Dropdown(options=predictores, value="Cement", description="Variable"),
    fraccion=widgets.FloatSlider(value=2/3, min=0.2, max=1, step=0.05,
                                description="Suavizado", continuous_update=False),
)
display(control_relacion)

#### Descriptiva bivariante para las ocho variables predictoras

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, variable in zip(axes.flat, predictores):
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable)
    _ = ax.set_title(variable)
fig.tight_layout()
plt.show()

## Ajuste del modelo lineal simple

In [ ]:
y = datos["Strength"]
X_simple = sm.add_constant(datos[["Cement"]], has_constant="add")
mod_lm0 = sm.OLS(y, X_simple, missing="raise").fit()
mod_lm0.summary()

In [ ]:
rejilla = pd.DataFrame({"Cement": np.linspace(datos["Cement"].min(), datos["Cement"].max(), 100)})
fig, ax = plt.subplots()
_ = ax.scatter(datos["Cement"], y, s=12, alpha=0.45)
_ = ax.plot(rejilla["Cement"], mod_lm0.predict(sm.add_constant(rejilla, has_constant="add")), color="red")
_ = ax.set(xlabel="Cement", ylabel="Strength", title="Modelo lineal simple")
plt.show()

## Ajuste del modelo multivariable

Incluimos las ocho variables predictoras. `const` es el intercepto.
La tabla de coeficientes y el resumen permiten inspeccionar el ajuste.

In [ ]:
X_lineal = sm.add_constant(datos[predictores], has_constant="add")
mod_lm1 = sm.OLS(y, X_lineal, missing="raise").fit()
mod_lm1.params

In [ ]:
mod_lm1.summary()

## Selección de variables

Partimos del modelo completo y retiramos en cada paso el término cuya eliminación reduce más el AIC.
Nos detenemos cuando ninguna retirada lo mejora. Los términos con varios coeficientes se seleccionan como un grupo.

In [ ]:
def seleccion_atras(respuesta, matriz, grupos):
    activas = list(grupos)

    def ajustar(variables):
        columnas = ["const"] + [columna for variable in variables for columna in grupos[variable]]
        return sm.OLS(respuesta, matriz[columnas], missing="raise").fit()

    with threadpool_limits(limits=1):
        modelo = ajustar(activas)
        pasos = [{"Paso": 0, "Eliminada": "—", "AIC": modelo.aic, "Términos": len(activas)}]
        while activas:
            candidatos = []
            for variable in activas:
                restantes = [nombre for nombre in activas if nombre != variable]
                candidato = ajustar(restantes)
                candidatos.append((candidato.aic, variable, candidato))
            aic, retirada, candidato = min(candidatos, key=lambda resultado: resultado[0])
            if aic >= modelo.aic - 1e-7:
                break
            activas.remove(retirada)
            modelo = candidato
            pasos.append({"Paso": len(pasos), "Eliminada": retirada, "AIC": aic, "Términos": len(activas)})
    return modelo, activas, pd.DataFrame(pasos)

In [ ]:
grupos_lineales = {variable: [variable] for variable in predictores}
mod_lm2, seleccion_lineal, pasos_lineales = seleccion_atras(y, X_lineal, grupos_lineales)
pasos_lineales

In [ ]:
mod_lm2.summary()

## Validación del modelo lineal

Examinamos residuos frente a ajustados, normalidad con un gráfico Q–Q,
dispersión de los residuos y relación entre residuos y apalancamiento.
Las líneas discontinuas del último panel corresponden a distancias de Cook de 0.5 y 1.

In [ ]:
def diagnosticos(modelo, titulo):
    influencia = modelo.get_influence()
    ajustados = np.asarray(modelo.fittedvalues)
    residuos = np.asarray(modelo.resid)
    estandarizados = influencia.resid_studentized_internal
    leverage = influencia.hat_matrix_diag
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    axes[0, 0].scatter(ajustados, residuos, s=12, alpha=0.45)
    curva = lowess(residuos, ajustados, frac=2/3, it=3)
    axes[0, 0].plot(curva[:, 0], curva[:, 1], color="red")
    axes[0, 0].axhline(0, color="grey", linestyle="--")
    axes[0, 0].set(title="Residuos frente a ajustados", xlabel="Ajustados", ylabel="Residuos")

    sm.ProbPlot(estandarizados).qqplot(line="45", ax=axes[0, 1], marker="o", markersize=3, alpha=0.45)
    axes[0, 1].set(title="Q–Q de residuos estandarizados", xlabel="Cuantiles normales", ylabel="Residuos estandarizados")

    escala = np.sqrt(np.abs(estandarizados))
    axes[1, 0].scatter(ajustados, escala, s=12, alpha=0.45)
    curva = lowess(escala, ajustados, frac=2/3, it=3)
    axes[1, 0].plot(curva[:, 0], curva[:, 1], color="red")
    axes[1, 0].set(title="Escala y localización", xlabel="Ajustados", ylabel="Raíz de |residuo estandarizado|")

    axes[1, 1].scatter(leverage, estandarizados, s=12, alpha=0.45)
    axes[1, 1].axhline(0, color="grey", linestyle="--")
    h = np.linspace(max(leverage.min(), 1e-5), leverage.max() * 1.05, 200)
    for distancia in [0.5, 1]:
        limite = np.sqrt(distancia * len(modelo.params) * (1 - h) / h)
        axes[1, 1].plot(h, limite, "--", color="red", alpha=0.6, label=f"Cook = {distancia}")
        axes[1, 1].plot(h, -limite, "--", color="red", alpha=0.6)
    axes[1, 1].set(title="Residuos y apalancamiento", xlabel="Apalancamiento", ylabel="Residuo estandarizado",
                   ylim=(estandarizados.min() - 1, estandarizados.max() + 1))
    axes[1, 1].legend(fontsize=8)
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()

In [ ]:
diagnosticos(mod_lm2, "Diagnóstico del modelo lineal seleccionado")

## Nueva descriptiva

### Residuos frente a las variables predictoras

Buscamos patrones que sugieran relaciones pendientes de modelizar.

In [ ]:
def residuos_predictoras(modelo, tabla, variables):
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, variable in zip(axes.flat, variables):
        dibujar_relacion(ax, tabla[variable], modelo.resid, variable)
        ax.set(title=variable, ylabel="Residuos")
        ax.axhline(0, color="grey", linestyle="--")
    for ax in list(axes.flat)[len(variables):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()

residuos_predictoras(mod_lm2, datos, predictores)

### Gráficos de residuos y pruebas de curvatura

Para los términos lineales, añadimos un término cuadrático y contrastamos si aporta información.
También incluimos los ajustados: su prueba de curvatura es el test de no aditividad de Tukey.
Los términos que ya contienen un polinomio de grado 2 se exploran gráficamente.

In [ ]:
def residuos_por_termino(modelo, tabla, grupos):
    variables = list(grupos)
    paneles = [(variable, tabla[variable], len(grupos[variable]) == 1) for variable in variables]
    paneles.append(("Ajustados (Tukey)", modelo.fittedvalues, True))
    filas = int(np.ceil(len(paneles) / 3))
    fig, axes = plt.subplots(filas, 3, figsize=(13, 3.5 * filas), squeeze=False)
    resultados = []
    for ax, (nombre, valores, probar) in zip(axes.flat, paneles):
        dibujar_relacion(ax, valores, modelo.resid, nombre)
        ax.set(title=nombre, ylabel="Residuos")
        ax.axhline(0, color="grey", linestyle="--")
        if probar:
            x = np.asarray(valores, dtype=float)
            x = (x - x.mean()) / x.std()
            ampliada = np.column_stack([modelo.model.exog, x ** 2])
            ampliado = sm.OLS(modelo.model.endog, ampliada).fit()
            f, p, gl = ampliado.compare_f_test(modelo)
            resultados.append({"Término": nombre, "F": f, "p-valor": p})
    for ax in list(axes.flat)[len(paneles):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()
    return pd.DataFrame(resultados).set_index("Término")

In [ ]:
residuos_por_termino(mod_lm2, datos, {v: grupos_lineales[v] for v in seleccion_lineal})

## Transformaciones polinómicas de las predictoras

Incluimos polinomios de grado 2 en `Cement`, `BlastFurnaceSlag`, `FlyAsh`, `Water`,
`Superplasticizer`, `FineAggregate` y `Age`. `CoarseAggregate` permanece lineal.

Usamos una base ortogonal de dos columnas por variable. La base se aprende con los datos de entrenamiento
y se conserva para transformar nuevas observaciones.

In [ ]:
class PolinomioOrtogonal:
    def __init__(self, valores):
        x = np.asarray(valores, dtype=float)
        self.centro = x.mean()
        self.escala = x.std()
        if self.escala == 0 or len(np.unique(x)) < 3:
            raise ValueError("Un polinomio de grado 2 necesita al menos tres valores distintos.")
        z = (x - self.centro) / self.escala
        vandermonde = np.column_stack([np.ones(len(z)), z, z ** 2])
        q, r = np.linalg.qr(vandermonde)
        signos = np.where(np.diag(r) < 0, -1, 1)
        r = signos[:, None] * r
        self.inversa = np.linalg.inv(r)

    def transformar(self, valores):
        x = np.asarray(valores, dtype=float)
        z = (x - self.centro) / self.escala
        vandermonde = np.column_stack([np.ones(len(z)), z, z ** 2])
        return (vandermonde @ self.inversa)[:, 1:]

def diseno_polinomico(tabla, bases=None):
    bases = {} if bases is None else dict(bases)
    matriz = pd.DataFrame({"const": np.ones(len(tabla))}, index=tabla.index)
    for variable in predictores:
        if variable == "CoarseAggregate":
            matriz[variable] = tabla[variable]
        else:
            if variable not in bases:
                bases[variable] = PolinomioOrtogonal(tabla[variable])
            matriz[[variable + "_1", variable + "_2"]] = bases[variable].transformar(tabla[variable])
    return matriz, bases

In [ ]:
X_polinomico, bases_polinomicas = diseno_polinomico(datos)
grupos_polinomicos = {
    variable: [variable] if variable == "CoarseAggregate" else [variable + "_1", variable + "_2"]
    for variable in predictores
}
mod_lm4 = sm.OLS(y, X_polinomico).fit()
mod_lm4.summary()

In [ ]:
mod_lm5, seleccion_polinomica, pasos_polinomicos = seleccion_atras(y, X_polinomico, grupos_polinomicos)
pasos_polinomicos

In [ ]:
mod_lm5.summary()

## Colinealidad

Para términos de varios coeficientes calculamos el GVIF y su versión ajustada por grados de libertad.
El VIF habitual corresponde a términos de un coeficiente; valores superiores a 5 son una referencia para revisar
la redundancia. La columna ajustada `GVIF^(1/(2·Df))` permite interpretar términos de distinta dimensión.

In [ ]:
def gvif(modelo, grupos):
    covarianza = modelo.cov_params().drop(index="const", columns="const")
    desviaciones = np.sqrt(np.diag(covarianza))
    correlacion = covarianza / np.outer(desviaciones, desviaciones)
    columnas = list(correlacion.columns)

    def logdet(matriz):
        if matriz.size == 0:
            return 0.0
        signo, valor = np.linalg.slogdet(matriz)
        if signo <= 0:
            raise ValueError("La matriz de coeficientes es singular.")
        return valor

    total = logdet(correlacion.to_numpy())
    resultados = []
    for variable, terminos in grupos.items():
        dentro = [columnas.index(termino) for termino in terminos]
        fuera = [i for i in range(len(columnas)) if i not in dentro]
        matriz = correlacion.to_numpy()
        valor = np.exp(logdet(matriz[np.ix_(dentro, dentro)]) + logdet(matriz[np.ix_(fuera, fuera)]) - total)
        gl = len(dentro)
        resultados.append({"Término": variable, "GVIF": valor, "Df": gl,
                           "GVIF^(1/(2·Df))": valor ** (1 / (2 * gl))})
    return pd.DataFrame(resultados).set_index("Término")

In [ ]:
grupos_seleccionados = {variable: grupos_polinomicos[variable] for variable in seleccion_polinomica}
gvif(mod_lm5, grupos_seleccionados)

## Validación del modelo polinómico

In [ ]:
diagnosticos(mod_lm5, "Diagnóstico del modelo polinómico seleccionado")

In [ ]:
residuos_por_termino(mod_lm5, datos, grupos_seleccionados)

## Transformación Box–Cox de la respuesta

Estimamos lambda maximizando la verosimilitud del modelo, con sus predictoras incluidas.
El perfil permite explorar qué potencias son compatibles con los datos.

In [ ]:
def logverosimilitud_boxcox(valor_lambda, modelo):
    respuesta = np.asarray(modelo.model.endog, dtype=float)
    if (respuesta <= 0).any():
        raise ValueError("Box–Cox requiere una respuesta estrictamente positiva.")
    transformada = boxcox(respuesta, valor_lambda)
    coeficientes = modelo.model.pinv_wexog @ transformada
    residuos = transformada - modelo.model.exog @ coeficientes
    n = len(respuesta)
    return -n / 2 * np.log(np.sum(residuos ** 2) / n) + (valor_lambda - 1) * np.log(respuesta).sum()

optimo_lambda = minimize_scalar(lambda valor: -logverosimilitud_boxcox(valor, mod_lm5),
                                bracket=(-1, 1), method="brent")
if not optimo_lambda.success:
    raise RuntimeError("No se ha podido estimar lambda.")
lamb = optimo_lambda.x
bc = pd.DataFrame({"lambda": np.linspace(-2, 2, 201)})
bc["logLik"] = [logverosimilitud_boxcox(valor, mod_lm5) for valor in bc["lambda"]]
lamb

In [ ]:
def explorar_boxcox(valor_lambda):
    fig, ax = plt.subplots()
    ax.plot(bc["lambda"], bc["logLik"], color="steelblue")
    ax.axvline(lamb, color="black", linestyle="--", label=f"Lambda estimada: {lamb:.3f}")
    ax.axvline(valor_lambda, color="red", label=f"Lambda seleccionada: {valor_lambda:.2f}")
    corte = logverosimilitud_boxcox(lamb, mod_lm5) - stats.chi2.ppf(0.95, 1) / 2
    ax.axhline(corte, color="grey", linestyle=":", label="Corte del intervalo del 95 %")
    ax.set(xlabel="Lambda", ylabel="Log-verosimilitud", title="Perfil Box–Cox")
    ax.legend(fontsize=8)
    plt.show()

In [ ]:
control_boxcox = widgets.interactive(
    explorar_boxcox,
    valor_lambda=widgets.FloatSlider(value=float(np.clip(lamb, -2, 2)), min=-2, max=2, step=0.05,
                                    description="Lambda", continuous_update=False),
)
display(control_boxcox)

### Ajustar el modelo con la respuesta transformada

El ajuste utiliza la estimación `lamb`. Creamos `Strength2` elevando `Strength` a esa potencia.
Si lambda es cero, usamos el logaritmo. Las predictoras mantienen sus términos polinómicos;
`CoarseAggregate` no se incluye en este ajuste.

In [ ]:
def transformar_respuesta(valores, valor_lambda):
    return np.log(valores) if abs(valor_lambda) < 1e-8 else np.power(valores, valor_lambda)

datos["Strength2"] = transformar_respuesta(datos["Strength"], lamb)
variables_sin_coarse = [variable for variable in predictores if variable != "CoarseAggregate"]
columnas_sin_coarse = ["const"] + [columna for variable in variables_sin_coarse
                                   for columna in grupos_polinomicos[variable]]
mod_lm6 = sm.OLS(datos["Strength2"], X_polinomico[columnas_sin_coarse]).fit()
mod_lm6.summary()

## Validación del modelo con respuesta transformada

In [ ]:
diagnosticos(mod_lm6, "Diagnóstico del modelo con Strength2")

### Para practicar en casa

¿Qué ocurre si aplicas transformaciones logarítmicas a las predictoras?
Prueba tus propuestas y revisa sus diagnósticos. Comprueba el dominio de cada transformación.

In [ ]:
# Tu solución

## Observaciones influyentes

Revisamos los residuos studentizados, sus p-valores ajustados por Bonferroni,
el apalancamiento y la distancia de Cook. El análisis se realiza sobre `mod_lm5`.

In [ ]:
with threadpool_limits(limits=1):
    influencia_modelo = mod_lm5.get_influence()
    studentizados = influencia_modelo.resid_studentized_external

influencia = pd.DataFrame({
    "Cook": influencia_modelo.cooks_distance[0],
    "Residuo studentizado": studentizados,
    "p Bonferroni": np.minimum(1, 2 * stats.t.sf(np.abs(studentizados), mod_lm5.df_resid - 1) * len(datos)),
    "Apalancamiento": influencia_modelo.hat_matrix_diag,
}, index=datos.index)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, columna in zip(axes.flat, influencia.columns):
    _ = ax.vlines(influencia.index, 0, influencia[columna], color="steelblue", alpha=0.65)
    _ = ax.set(xlabel="Observación", ylabel=columna, title=columna)
    for observacion in [65, 81, 147, 248]:
        _ = ax.annotate(str(observacion), (observacion, influencia.loc[observacion, columna]),
                        xytext=(3, 4), textcoords="offset points", fontsize=8)
fig.tight_layout()
plt.show()

### Residuos, apalancamiento e influencia

El tamaño de las burbujas representa la distancia de Cook.
Explora las observaciones 81 y 147, la observación 248 con un residuo destacado,
y el apalancamiento de las observaciones 65 y 81.

In [ ]:
def grafico_influencia(etiquetas):
    fig, ax = plt.subplots(figsize=(9, 6))
    tamanos = 20 + 800 * influencia["Cook"] / influencia["Cook"].max()
    ax.scatter(influencia["Apalancamiento"], influencia["Residuo studentizado"],
               s=tamanos, alpha=0.4, edgecolors="black")
    ax.axhline(0, color="grey", linestyle="--")
    for valor in [-2, 2]:
        ax.axhline(valor, color="grey", linestyle=":")
    for factor in [2, 3]:
        ax.axvline(factor * len(mod_lm5.params) / len(datos), color="red", linestyle=":")
    if etiquetas:
        for observacion in [65, 81, 147, 248]:
            fila = influencia.loc[observacion]
            ax.annotate(str(observacion), (fila["Apalancamiento"], fila["Residuo studentizado"]),
                        xytext=(6, 5), textcoords="offset points")
    ax.set(xlabel="Apalancamiento", ylabel="Residuo studentizado", title="Influencia de las observaciones")
    plt.show()

In [ ]:
control_influencia = widgets.interactive(
    grafico_influencia,
    etiquetas=widgets.Checkbox(value=True, description="Mostrar observaciones señaladas"),
)
display(control_influencia)

#### Inspeccionar las observaciones seleccionadas

In [ ]:
obs_rm = [81, 147, 248]
datos.loc[obs_rm]

#### Localizarlas en los gráficos de pares

In [ ]:
col_points = np.array(["#0000001a"] * len(datos), dtype=object)
for observacion, color in zip(obs_rm, ["red", "green", "blue"]):
    col_points[datos.index.get_loc(observacion)] = color

_ = scatter_matrix(datos[columnas_originales], figsize=(14, 14), diagonal="hist",
                   c=col_points, s=12, alpha=None)
plt.show()

### Ajustar sin las observaciones señaladas

Creamos `datos_rm` excluyendo las observaciones 81, 147 y 248.
Reajustamos los siete términos polinómicos con las observaciones restantes y examinamos ambos resúmenes.

In [ ]:
datos_rm = datos.drop(index=obs_rm).copy()
X_rm, bases_rm = diseno_polinomico(datos_rm)
mod_lm7 = sm.OLS(datos_rm["Strength"], X_rm[columnas_sin_coarse]).fit()
mod_lm7.summary()

In [ ]:
mod_lm5.summary()

### Modelo final

El modelo final es `mod_lm5`, ajustado con el conjunto completo de entrenamiento.

In [ ]:
mod_final = mod_lm5

### Efectos

Representamos la resistencia predicha al variar cada predictor. Los términos de las demás variables
se fijan en su media en la matriz del modelo. Las bandas muestran el intervalo de confianza del 95 %
para la respuesta media.

In [ ]:
def datos_efecto(variable):
    valores = np.linspace(datos[variable].min(), datos[variable].max(), 100)
    columnas = mod_final.model.exog_names
    medias = mod_final.model.exog.mean(axis=0)
    matriz = pd.DataFrame(np.tile(medias, (len(valores), 1)), columns=columnas)
    if variable in bases_polinomicas:
        matriz[grupos_polinomicos[variable]] = bases_polinomicas[variable].transformar(valores)
    else:
        matriz[variable] = valores
    resumen = mod_final.get_prediction(matriz).summary_frame(alpha=0.05)
    return valores, resumen

def dibujar_efecto(ax, variable, banda=True):
    valores, resumen = datos_efecto(variable)
    ax.plot(valores, resumen["mean"].to_numpy(), color="steelblue")
    if banda:
        ax.fill_between(valores, resumen["mean_ci_lower"].to_numpy(), resumen["mean_ci_upper"].to_numpy(),
                        color="steelblue", alpha=0.2)
    ax.set(xlabel=variable, ylabel="Strength predicha", title=variable)

In [ ]:
filas_efectos = int(np.ceil(len(seleccion_polinomica) / 4))
fig, axes = plt.subplots(filas_efectos, 4, figsize=(16, 4 * filas_efectos), squeeze=False)
for ax, variable in zip(axes.flat, seleccion_polinomica):
    dibujar_efecto(ax, variable)
for ax in list(axes.flat)[len(seleccion_polinomica):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

Elige un predictor para examinar su efecto con más detalle.

In [ ]:
def explorar_efecto(variable, banda):
    fig, ax = plt.subplots()
    dibujar_efecto(ax, variable, banda)
    plt.show()

In [ ]:
control_efectos = widgets.interactive(
    explorar_efecto,
    variable=widgets.Dropdown(options=seleccion_polinomica, description="Variable"),
    banda=widgets.Checkbox(value=True, description="Intervalo del 95 %"),
)
display(control_efectos)

## Evaluar con nuevos datos

### Leer el conjunto de test

Creamos también `Strength2` con la lambda aprendida en entrenamiento.
El modelo final predice `Strength` en su escala original.

In [ ]:
test = pd.read_csv(DATA_PATH / "Concrete_test.txt", sep="\t")
test.index = np.arange(1, len(test) + 1)
test.index.name = "observacion"
test["Strength2"] = transformar_respuesta(test["Strength"], lamb)

### Predicciones

Transformamos las predictoras utilizando las bases polinómicas guardadas durante el entrenamiento.

In [ ]:
X_test, _ = diseno_polinomico(test, bases_polinomicas)
pr = mod_final.predict(X_test[mod_final.model.exog_names]).rename("Prediccion")
pr

### Predicciones y valores observados

La diagonal representa coincidencia entre predicción y observación.
Selecciona una observación de test para localizarla y consultar sus valores.

In [ ]:
def explorar_prediccion(observacion):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(pr, test["Strength"], s=15, alpha=0.45)
    minimo = min(pr.min(), test["Strength"].min())
    maximo = max(pr.max(), test["Strength"].max())
    ax.plot([minimo, maximo], [minimo, maximo], color="red", linewidth=2)
    ax.scatter(pr.loc[observacion], test.loc[observacion, "Strength"], s=65, color="orange", edgecolors="black", zorder=3)
    ax.annotate(str(observacion), (pr.loc[observacion], test.loc[observacion, "Strength"]),
                xytext=(6, 6), textcoords="offset points")
    ax.set(xlabel="Predicción", ylabel="Strength observada", title="Predicciones en test")
    ax.set_aspect("equal", adjustable="box")
    plt.show()
    display(test.loc[[observacion], columnas_originales].assign(
        Prediccion=pr.loc[observacion], Residuo=test.loc[observacion, "Strength"] - pr.loc[observacion],
    ))

In [ ]:
control_predicciones = widgets.interactive(
    explorar_prediccion,
    observacion=widgets.IntSlider(value=1, min=1, max=len(test), description="Observación",
                                  continuous_update=False, style={"description_width": "initial"}),
)
display(control_predicciones)

### Error cuadrático medio

El EQM promedia los errores de predicción al cuadrado.
Su raíz (RMSE) expresa el tamaño del error en las unidades de la resistencia.

In [ ]:
n = test.shape[0]
EQM = ((pr - test["Strength"]) ** 2).sum() / n
EQM

In [ ]:
RMSE = np.sqrt(EQM)
RMSE

La desviación típica observada describe la variabilidad de la resistencia antes de usar las predicciones.

In [ ]:
test["Strength"].std(ddof=1)

In [ ]:
test["Strength"].describe()

### Documentación de las herramientas

- [Regresión por mínimos cuadrados en statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).
- [Suavizado LOWESS](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html).
- [Transformación Box–Cox en SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.boxcox.html).